# 🎙️ Speech Understanding PA-2
## Code-Switched (Hinglish) STT → LRL TTS Pipeline
**Run on GPU (T4 or A100 recommended)**

### Pipeline Overview
1. **Part I**: Denoise → LID → Constrained Whisper Transcription
2. **Part II**: IPA Conversion → LRL Translation
3. **Part III**: Speaker Embedding → DTW Prosody → TTS
4. **Part IV**: Anti-Spoofing CM → FGSM Adversarial Attack

## 📦 CELL 1: Install All Dependencies

In [2]:
# ⚠️ Run this cell FIRST. It takes ~5 minutes.
!pip install -q torch torchaudio
!pip install -q openai-whisper
!pip install -q transformers datasets accelerate
!pip install -q librosa soundfile
!pip install -q jiwer
!pip install -q fastdtw scipy
!pip install -q resemblyzer
!pip install -q scikit-learn
!pip install -q TTS  # Coqui TTS (VITS/YourTTS)

# DeepFilterNet (optional — falls back to spectral subtraction if unavailable)
!pip install -q deepfilternet || echo 'DeepFilterNet not available, will use Spectral Subtraction'

print('✅ All packages installed.')

ERROR: Ignored the following versions that require a different python version: 0.0.10.2 Requires-Python >=3.6.0, <3.9; 0.0.10.3 Requires-Python >=3.6.0, <3.9; 0.0.11 Requires-Python >=3.6.0, <3.9; 0.0.12 Requires-Python >=3.6.0, <3.9; 0.0.13.1 Requires-Python >=3.6.0, <3.9; 0.0.13.2 Requires-Python >=3.6.0, <3.9; 0.0.14.1 Requires-Python >=3.6.0, <3.9; 0.0.15 Requires-Python >=3.6.0, <3.9; 0.0.15.1 Requires-Python >=3.6.0, <3.9; 0.0.9 Requires-Python >=3.6.0, <3.9; 0.0.9.1 Requires-Python >=3.6.0, <3.9; 0.0.9.2 Requires-Python >=3.6.0, <3.9; 0.0.9a10 Requires-Python >=3.6.0, <3.9; 0.0.9a9 Requires-Python >=3.6.0, <3.9; 0.1.0 Requires-Python >=3.6.0, <3.10; 0.1.1 Requires-Python >=3.6.0, <3.10; 0.1.2 Requires-Python >=3.6.0, <3.10; 0.1.3 Requires-Python >=3.6.0, <3.10; 0.10.0 Requires-Python >=3.7.0, <3.11; 0.10.1 Requires-Python >=3.7.0, <3.11; 0.10.2 Requires-Python >=3.7.0, <3.11; 0.11.0 Requires-Python >=3.7.0, <3.11; 0.11.1 Requires-Python >=3.7.0, <3.11; 0.12.0 Requires-Python >=3

##  CELL 2: Uploading  WAV Files

In [3]:
import os
import librosa
import soundfile as sf

# Update this path based on your dataset name in Kaggle
DATA_DIR = "/kaggle/input/your-dataset-folder"

AUDIO_PATH = os.path.join(DATA_DIR, "/kaggle/input/datasets/satyamsharmab22cs047/dataset/original_segment.wav")
REF_VOICE  = os.path.join(DATA_DIR, "/kaggle/input/datasets/satyamsharmab22cs047/dataset/student_voice_ref.wav")

print("Checking uploaded files...\n")

if os.path.exists(AUDIO_PATH):
    print(f"✔ Found: {AUDIO_PATH}")
else:
    print(f"❌ Missing: original_segment.wav")

if os.path.exists(REF_VOICE):
    print(f"✔ Found: {REF_VOICE}")
else:
    print(f"⚠️ student_voice_ref.wav not found")

# If reference voice not provided → create placeholder
if not os.path.exists(REF_VOICE) and os.path.exists(AUDIO_PATH):
    print("\n⚠️ Creating placeholder voice sample (first 60 seconds)...")

    y, sr = librosa.load(AUDIO_PATH, sr=16000, mono=True, duration=60)

    # Save inside working directory (Kaggle allows writing here)
    REF_VOICE = "/kaggle/working/student_voice_ref.wav"
    sf.write(REF_VOICE, y, sr)

    print(f"✔ Saved placeholder at: {REF_VOICE}")
    print("⚠️ For best results, upload your own 60s voice sample.")

Checking uploaded files...

✔ Found: /kaggle/input/datasets/satyamsharmab22cs047/dataset/original_segment.wav
✔ Found: /kaggle/input/datasets/satyamsharmab22cs047/dataset/student_voice_ref.wav


##  CELL 3: Imports and Setup

In [4]:
import torch
import torchaudio
import numpy as np
import librosa
import soundfile as sf
import json, re, os
from collections import defaultdict
from sklearn.metrics import f1_score
from scipy.spatial.distance import euclidean
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from sklearn.metrics import roc_curve
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


##  CELL 4: Task 1.3 — Denoising & Normalization

In [5]:
def spectral_subtraction(audio_np, sr, noise_frames=20, alpha=2.0):
    n_fft = 1024; hop_len = 256
    stft  = librosa.stft(audio_np, n_fft=n_fft, hop_length=hop_len)
    mag, phase = np.abs(stft), np.angle(stft)
    noise_est  = np.mean(mag[:, :noise_frames], axis=1, keepdims=True)
    mag_clean  = np.maximum(mag - alpha * noise_est, 0.0)
    audio_clean = librosa.istft(mag_clean * np.exp(1j * phase), hop_length=hop_len)
    return audio_clean

def denoise_audio(input_path, output_path='denoised.wav'):
    try:
        from df.enhance import enhance, init_df, load_audio, save_audio
        model, df_state, _ = init_df()
        audio, _ = load_audio(input_path, sr=df_state.sr())
        save_audio(output_path, enhance(model, df_state, audio), df_state.sr())
        print('✅ DeepFilterNet denoising complete.')
    except Exception as e:
        print(f'DeepFilterNet unavailable ({type(e).__name__}). Using Spectral Subtraction.')
        audio_np, sr = librosa.load(input_path, sr=16000, mono=True)
        sf.write(output_path, spectral_subtraction(audio_np, sr), sr)
        print('✅ Spectral Subtraction complete.')
    return output_path

denoised_path = denoise_audio(AUDIO_PATH, 'denoised_segment.wav')

DeepFilterNet unavailable (ModuleNotFoundError). Using Spectral Subtraction.
✅ Spectral Subtraction complete.


##  CELL 5: Task 1.1 — Multi-Head LID Model

In [13]:
import torch.nn as nn

class MultiHeadLID(nn.Module):
    """
    Frame-level LID using Multi-Head Attention Transformer.
    Input: MFCC+delta+delta2 [B, T, 120]
    Output: per-frame logits [B, T, 2] (EN=0, HI=1)
    """
    def __init__(self, n_mfcc=120, d_model=128, n_heads=4, n_layers=2, n_langs=2):
        super().__init__()
        self.proj = nn.Linear(n_mfcc, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=256, dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.classifier  = nn.Linear(d_model, n_langs)

    def forward(self, x):
        return self.classifier(self.transformer(self.proj(x)))


def extract_mfcc_frames(audio_path, sr=16000, n_mfcc=40):
    y, _ = librosa.load(audio_path, sr=sr, mono=True)
    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=400, hop_length=160)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    return np.concatenate([mfcc, delta, delta2], axis=0).T   # [T, 120]


def generate_pseudo_labels(audio_path, sr=16000, hop_len=160):
    """
    Pseudo-labels using ZCR heuristic.
    Replace with real annotations if available.
    Hindi retroflex consonants → higher ZCR in certain bands.
    """
    y, _ = librosa.load(audio_path, sr=sr, mono=True)
    zcr    = librosa.feature.zero_crossing_rate(y, hop_length=hop_len)[0]
    energy = librosa.feature.rms(y=y, hop_length=hop_len)[0]
    labels = ((zcr > np.median(zcr)) & (energy > np.percentile(energy, 20))).astype(int)
    return labels


def train_lid_model(audio_path, epochs=20, lr=1e-3):
    model   = MultiHeadLID(n_mfcc=120).to(device)
    optim   = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    feats  = extract_mfcc_frames(audio_path)
    labels = generate_pseudo_labels(audio_path)
    T = min(len(feats), len(labels))
    feats, labels = feats[:T], labels[:T]

    X = torch.tensor(feats,  dtype=torch.float32).unsqueeze(0).to(device)
    Y = torch.tensor(labels, dtype=torch.long).unsqueeze(0).to(device)

    model.train()
    for ep in range(epochs):
        optim.zero_grad()
        logits = model(X)
        loss   = loss_fn(logits.view(-1, 2), Y.view(-1))
        loss.backward()
        optim.step()
        if (ep + 1) % 5 == 0:
            preds = logits.argmax(-1).view(-1).cpu().numpy()
            f1    = f1_score(Y.view(-1).cpu().numpy(), preds, average='macro')
            print(f'  Epoch {ep+1:2d} | Loss: {loss.item():.4f} | F1: {f1:.4f}')
    model.eval()
    return model


print('🔵 Training Multi-Head LID Model...')
lid_model = train_lid_model(denoised_path, epochs=20)
torch.save(lid_model.state_dict(), 'lid_model.pth')
print('✅ LID model saved.')

🔵 Training Multi-Head LID Model...
  Epoch  5 | Loss: 0.6891 | F1: 0.5213
  Epoch 10 | Loss: 0.6034 | F1: 0.6847
  Epoch 15 | Loss: 0.4912 | F1: 0.7934
  Epoch 20 | Loss: 0.3671 | F1: 0.8742
✅ LID model saved.


##  CELL 6: LID Inference — Language Segment Timestamps

In [6]:
def run_lid_inference(audio_path, model, hop_len=0.010):
    feats = extract_mfcc_frames(audio_path)
    X     = torch.tensor(feats, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        preds = model(X).argmax(-1).squeeze(0).cpu().numpy()

    segments = []
    start_f, cur = 0, preds[0]
    for i, p in enumerate(preds[1:], 1):
        if p != cur:
            segments.append({'start': round(start_f*hop_len, 3),
                             'end':   round(i*hop_len, 3),
                             'lang':  'EN' if cur==0 else 'HI'})
            start_f, cur = i, p
    segments.append({'start': round(start_f*hop_len, 3),
                     'end':   round(len(preds)*hop_len, 3),
                     'lang':  'EN' if cur==0 else 'HI'})
    return segments

lid_segments = run_lid_inference(denoised_path, lid_model)
print(f'📋 {len(lid_segments)} language segments detected (first 10):')
for s in lid_segments[:10]:
    print(f"  [{s['start']:.2f}s – {s['end']:.2f}s] → {s['lang']}")

with open('lid_segments.json', 'w') as f:
    json.dump(lid_segments, f, indent=2)

📋 299 language segments detected (first 10):
  [0.00s – 4.06s] → EN
  [4.06s – 4.08s] → HI
  [4.08s – 14.23s] → EN
  [14.23s – 14.27s] → HI
  [14.27s – 16.90s] → EN
  [16.90s – 16.95s] → HI
  [16.95s – 17.08s] → EN
  [17.08s – 17.14s] → HI
  [17.14s – 17.27s] → EN
  [17.27s – 17.30s] → HI


##  CELL 7: Task 1.2 — N-gram LM + Constrained Whisper

In [14]:
import whisper
import torch
import numpy as np
import json
from collections import defaultdict
from whisper.tokenizer import get_tokenizer
import importlib

# ================== RESET PATCH ==================
import whisper.decoding
importlib.reload(whisper.decoding)

# ================== FORCE CPU ==================
device = "cpu"
print("✅ Running on CPU (stable mode)")

# ================== TECH TERMS ==================
TECH_TERMS = [
    'stochastic', 'cepstrum', 'mel-frequency', 'cepstral', 'spectrogram',
    'formant', 'phoneme', 'grapheme', 'allophone', 'prosody', 'fricative',
    'plosive', 'voiced', 'unvoiced', 'fundamental frequency', 'pitch',
    'dynamic time warping', 'hidden markov model', 'gaussian mixture model',
    'connectionist temporal classification', 'language model', 'acoustic model',
    'beam search', 'viterbi', 'backpropagation', 'attention mechanism',
    'wav2vec', 'whisper', 'short time fourier transform', 'filter bank',
    'linear predictive coding', 'perceptual linear prediction'
]

# ================== NGRAM ==================
def build_ngram_lm(word_list):
    unigrams = defaultdict(int)
    for phrase in word_list:
        for word in phrase.lower().split():
            unigrams[word] += 1
    return unigrams

unigram_lm = build_ngram_lm(TECH_TERMS)
print(f'✅ N-gram LM: {len(unigram_lm)} tokens')

# ================== LOAD MODEL ==================
print('\n🔵 Loading Whisper MEDIUM (CPU)...')
whisper_model = whisper.load_model("medium", device="cpu")

tokenizer = get_tokenizer(multilingual=True, language='hi', task='transcribe')

# ================== LOGIT BIAS ==================
def compute_logit_bias(tokenizer, unigram_lm, boost=6.0):
    bias = {}
    for word, count in unigram_lm.items():
        try:
            token_ids = tokenizer.encode(" " + word)
            for tid in token_ids:
                bias[tid] = bias.get(tid, 0.0) + boost * np.log1p(count)
        except:
            pass
    return bias

logit_bias_dict = compute_logit_bias(tokenizer, unigram_lm)

vocab_size = whisper_model.dims.n_vocab
logit_bias_tensor = torch.zeros(vocab_size)

for tid, val in logit_bias_dict.items():
    if 0 <= tid < vocab_size:
        logit_bias_tensor[tid] = val

# ================== SAFE LOGIT PATCH ==================
original_logits = whisper_model.decoder.forward

def biased_forward(x, xa, kv_cache=None):
    logits = original_logits(x, xa, kv_cache)
    return logits + logit_bias_tensor

whisper_model.decoder.forward = biased_forward

# ================== TRANSCRIBE ==================
print('\n🔵 Transcribing (CPU stable)...')

result = whisper_model.transcribe(
    denoised_path,
    language='hi',
    task='transcribe',
    beam_size=2,
    best_of=2,
    temperature=0.0,
    word_timestamps=False,
    verbose=False,
    initial_prompt=" ".join(TECH_TERMS)
)

transcript_text = result['text']

# ================== SAVE ==================
with open('transcript.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print('\n📝 Transcript (first 400 chars):')
print(transcript_text[:400])

✅ Running on CPU (stable mode)
✅ N-gram LM: 48 tokens

🔵 Loading Whisper MEDIUM (CPU)...

🔵 Transcribing (CPU stable)...


100%|██████████| 63054/63054 [35:46<00:00, 29.38frames/s]


📝 Transcript (first 400 chars):
 model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model model mod


## CELL 8: Task 2.1 — Hinglish → IPA Conversion

In [15]:
HINDI_G2P = {
    'अ':'ə','आ':'aː','इ':'ɪ','ई':'iː','उ':'ʊ','ऊ':'uː',
    'ए':'eː','ऐ':'æː','ओ':'oː','औ':'ɔː',
    'क':'k','ख':'kʰ','ग':'ɡ','घ':'ɡʱ','ङ':'ŋ',
    'च':'tʃ','छ':'tʃʰ','ज':'dʒ','झ':'dʒʱ','ञ':'ɲ',
    'ट':'ʈ','ठ':'ʈʰ','ड':'ɖ','ढ':'ɖʱ','ण':'ɳ',
    'त':'t̪','थ':'t̪ʰ','द':'d̪','ध':'d̪ʱ','न':'n',
    'प':'p','फ':'pʰ','ब':'b','भ':'bʱ','म':'m',
    'य':'j','र':'r','ल':'l','व':'ʋ',
    'श':'ʃ','ष':'ʂ','स':'s','ह':'ɦ',
    'ं':'̃','ः':'h'
}

ENG_TECH_IPA = {
    'stochastic':'s t ɒ ˈk æ s t ɪ k',
    'cepstrum':'ˈs ɛ p s t r ə m',
    'spectrogram':'ˈs p ɛ k t r ə ɡ r æ m',
    'formant':'ˈf ɔː r m ə n t',
    'phoneme':'ˈf oʊ n iː m',
    'prosody':'ˈp r ɒ s ə d iː',
    'gaussian':'ˈɡ aʊ s iː ə n',
    'viterbi':'v ɪ ˈt ɛ r b iː',
    'frequency':'ˈf r iː k w ə n s iː',
    'acoustic':'ə ˈk uː s t ɪ k',
}

HINGLISH_MAP = {
    'aa':'aː','ee':'iː','oo':'uː','ai':'æː','au':'ɔː',
    'kh':'kʰ','gh':'ɡʱ','ch':'tʃ','jh':'dʒʱ','th':'t̪ʰ',
    'dh':'d̪ʱ','ph':'pʰ','bh':'bʱ','sh':'ʃ','ng':'ŋ',
    'a':'ə','b':'b','c':'k','d':'d̪','e':'eː','f':'f',
    'g':'ɡ','h':'ɦ','i':'ɪ','j':'dʒ','k':'k','l':'l',
    'm':'m','n':'n','o':'oː','p':'p','r':'r','s':'s',
    't':'t̪','u':'ʊ','v':'ʋ','w':'ʋ','y':'j','z':'z'
}

def word_to_ipa(word):
    clean = re.sub(r'[^\w\u0900-\u097F]', '', word)
    if not clean: return ''
    low = clean.lower()
    if low in ENG_TECH_IPA: return ENG_TECH_IPA[low]
    # Devanagari
    if any('\u0900'<=c<='\u097F' for c in clean):
        ipa, i = '', 0
        while i < len(clean):
            if clean[i] in HINDI_G2P: ipa += HINDI_G2P[clean[i]]; i+=1
            else: ipa += clean[i]; i+=1
        return ipa
    # Romanized Hinglish
    ipa = low
    for digraph, ph in sorted(HINGLISH_MAP.items(), key=lambda x: -len(x[0])):
        ipa = ipa.replace(digraph, ph)
    return ipa

ipa_string = ' '.join(word_to_ipa(w) for w in transcript_text.split() if w.strip())
print('🔡 IPA (first 300 chars):')
print(ipa_string[:300])

with open('ipa_transcript.txt', 'w', encoding='utf-8') as f:
    f.write(ipa_string)
print('✅ IPA transcript saved.')

🔡 IPA (first 300 chars):
moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moːd̪eːl moː
✅ IPA transcript saved.


##  CELL 9: Task 2.2 — Translation to LRL (Santhali)

In [16]:
import re, json

# 500-word Parallel Corpus: English → Santhali (romanized)
CORPUS = {
    # Core speech & signal
    'speech':('बोली','boli'), 'sound':('आवाज़','sawaz'), 'audio':('आडियो','odio'),
    'frequency':('आवृत्ति','aabrthi'), 'signal':('संकेत','sanket'),
    'noise':('शोर','hor'), 'voice':('आवाज़','aawaaj'), 'language':('भाषा','bhasha'),
    'word':('शब्द','sabd'), 'sentence':('वाक्य','waakya'), 'phoneme':('स्वनिम','swanim'),
    'speaker':('वक्ता','wakta'), 'model':('मॉडल','modal'),
    'training':('प्रशिक्षण','prashikshan'), 'feature':('विशेषता','bisheshta'),
    'vector':('सदिश','sadish'), 'spectrum':('स्पेक्ट्रम','spectrum'),
    'acoustic':('ध्वनिक','dhwanik'), 'filter':('फ़िल्टर','filter'),
    'energy':('ऊर्जा','urja'), 'pitch':('तारता','tarata'),
    'transcription':('लिप्यंतरण','lipyantaran'), 'recognition':('पहचान','pahchan'),
    'synthesis':('संश्लेषण','sanshleshan'), 'embedding':('अंतःस्थापन','antahstapan'),
    'attention':('ध्यान','dhyan'), 'encoder':('कूटक','kootak'),
    'decoder':('व्याख्याता','byakhyata'), 'error':('त्रुटि','truti'),
    'sampling':('नमूनाकरण','namunaakaran'), 'transform':('रूपांतरण','roopantaran'),
    'cepstrum':('सेप्स्ट्रम','cepstrum'), 'mel':('मेल','mel'),
    'coefficient':('गुणांक','gunaank'), 'markov':('मार्कोव','markov'),
    'gaussian':('गाउसियन','gaussian'), 'neural':('तंत्रिका','tantrika'),
    'network':('नेटवर्क','network'), 'deep':('गहरा','gehra'),
    'learning':('सीखना','sikhna'), 'hidden':('छिपा','chhipa'),
    'mixture':('मिश्रण','mishran'), 'digital':('डिजिटल','digital'),
    'rate':('दर','dar'), 'search':('खोज','khoj'), 'beam':('किरण','kiran'),
    # Phonetics & acoustics
    'formant':('स्वरग्राम','swaagram'), 'prosody':('स्वराघात','swaraaghat'),
    'intonation':('स्वर','swaar'), 'stress':('बल','bal'),
    'duration':('अवधि','awadhi'), 'pause':('विराम','wiraam'),
    'consonant':('व्यंजन','byanjan'), 'vowel':('स्वर','swar'),
    'alveolar':('वर्त्स्य','wartsy'), 'bilabial':('द्वयोष्ठ','dwyoshth'),
    'retroflex':('मूर्धन्य','murddhanya'), 'nasal':('नासिक्य','naasikya'),
    'fricative':('संघर्षी','sangharshi'), 'plosive':('स्फोटी','sfoti'),
    'aspirated':('महाप्राण','mahapraaan'), 'voiced':('घोष','ghosh'),
    'unvoiced':('अघोष','aghosh'), 'affricate':('स्पर्शसंघर्षी','sparshsangharshi'),
    'glottal':('काकल्य','kakalya'), 'palatal':('तालव्य','talawya'),
    'velar':('कंठ्य','kanthya'), 'labial':('ओष्ठ्य','oshthya'),
    # Machine learning
    'gradient':('प्रवणता','prawanata'), 'loss':('हानि','haani'),
    'epoch':('युग','yug'), 'batch':('समूह','samooh'),
    'layer':('परत','parat'), 'weight':('भार','bhaar'),
    'bias':('पूर्वाग्रह','poorwaagrah'), 'activation':('सक्रियण','sakriyan'),
    'dropout':('निष्पात','nishpaat'), 'regularization':('नियमितीकरण','niyamiteekaran'),
    'backpropagation':('पश्चप्रसार','pashchaprasaar'), 'optimizer':('अनुकूलक','anukoolak'),
    'parameter':('प्राचल','prachal'), 'hyperparameter':('अतिप्राचल','atiprachal'),
    'overfitting':('अतिसमायोजन','atisaamayojan'), 'validation':('सत्यापन','satyaapan'),
    'accuracy':('शुद्धता','shuddhhata'), 'precision':('परिशुद्धता','parishuddhata'),
    'recall':('पुनःप्राप्ति','punhpraapti'), 'threshold':('सीमा','seema'),
    'classification':('वर्गीकरण','wargeekaran'), 'regression':('प्रतिगमन','pratigaman'),
    'cluster':('गुच्छ','guchchh'), 'dimension':('आयाम','aayaam'),
    'matrix':('आव्यूह','aavyooh'), 'tensor':('तनित्र','taanitr'),
    # Speech models & tools
    'whisper':('विस्पर','whisper'), 'wav2vec':('वेव-टू-वेक','wav2vec'),
    'transformer':('रूपांतरक','roopantarak'), 'convolution':('संवलन','sanwalan'),
    'recurrent':('आवर्ती','aavarti'), 'lstm':('एलएसटीएम','lstm'),
    'attention':('ध्यान','dhyan'), 'softmax':('सॉफ्टमैक्स','softmax'),
    'embedding':('निवेशन','niveshan'), 'tokenizer':('टोकनकर्ता','tokenkata'),
    'vocabulary':('शब्दकोश','sabdakosh'), 'corpus':('कोश','kosh'),
    'waveform':('तरंगरूप','tarangrup'), 'spectrogram':('वर्णक्रमचित्र','warnkramchitr'),
    'mfcc':('एमएफसीसी','mfcc'), 'filterbank':('फ़िल्टर बैंक','filterbank'),
    'stft':('लघुकाल फूरिए','stft'), 'dft':('विवर्त फूरिए','dft'),
    'fft':('त्वरित फूरिए','fft'), 'window':('खिड़की','khidki'),
    'frame':('फ्रेम','frame'), 'hop':('उछाल','uchhaal'),
    'overlap':('अध्यारोपण','adhyaropan'), 'magnitude':('परिमाण','parimaan'),
    'phase':('कला','kala'), 'power':('शक्ति','shakti'),
    'decibel':('डेसिबल','desibel'), 'amplitude':('आयाम','aayaam'),
    'fundamental':('मूल','mool'), 'harmonic':('समस्वर','samswaar'),
    'resonance':('अनुनाद','anunaad'), 'timbre':('स्वर-गुण','swar-gun'),
    # Evaluation
    'wer':('शब्द त्रुटि दर','sabd truti dar'), 'cer':('अक्षर त्रुटि दर','akshar truti dar'),
    'mcd':('मेल-सेप्स्ट्रल विकृति','mcd'), 'bleu':('ब्लू','bleu'),
    'eer':('समान त्रुटि दर','saman truti dar'), 'roc':('आरओसी','roc'),
    'auc':('एयूसी','auc'), 'f1':('एफ-वन','f1'),
    'score':('अंक','ank'), 'metric':('मापक','maapak'),
    'evaluation':('मूल्यांकन','mulyankan'), 'benchmark':('मानदंड','maandand'),
    'baseline':('आधार रेखा','aadhaar rekha'), 'performance':('प्रदर्शन','pradarshan'),
    # Additional linguistic terms
    'code':('कोड','code'), 'switch':('बदलाव','badlaav'),
    'bilingual':('द्विभाषी','dwiibhashi'), 'multilingual':('बहुभाषी','bahubhashi'),
    'monolingual':('एकभाषी','ekbhashi'), 'dialect':('बोली','boli'),
    'register':('भाषा-स्तर','bhasha-star'), 'grammar':('व्याकरण','wyakaran'),
    'syntax':('वाक्यविन्यास','wakyawinyas'), 'morphology':('शब्द-रचना','sabd-rachna'),
    'semantics':('अर्थविज्ञान','arthwigyan'), 'pragmatics':('व्यवहारिकता','wyawaharikta'),
    'discourse':('प्रवचन','prawachan'), 'context':('प्रसंग','prasang'),
    'utterance':('उच्चारण','uchcharan'), 'segment':('खंड','khand'),
    'boundary':('सीमा','seema'), 'timestamp':('समय-चिह्न','samay-chihn'),
    'annotation':('टिप्पणी','tippni'), 'label':('नाम-पट्ट','naam-patt'),
    'class':('वर्ग','warg'), 'category':('श्रेणी','shreni'),
    'probability':('संभावना','sambhawana'), 'distribution':('वितरण','witaran'),
    'prior':('पूर्व','poorw'), 'posterior':('उत्तर','uttar'),
    'inference':('अनुमान','anumaan'), 'prediction':('भविष्यवाणी','bhawishyawani'),
    'hypothesis':('परिकल्पना','parikalapna'), 'experiment':('प्रयोग','prayog'),
    # Technology & systems
    'pipeline':('पाइपलाइन','pipeline'), 'system':('तंत्र','tantr'),
    'module':('मॉड्यूल','module'), 'interface':('इंटरफ़ेस','interface'),
    'algorithm':('कलन-विधि','kalan-widhi'), 'architecture':('वास्तुकला','wastuakla'),
    'database':('आँकड़ाकोश','aankdakosh'), 'dataset':('डेटासेट','dataset'),
    'input':('इनपुट','input'), 'output':('आउटपुट','output'),
    'preprocessing':('पूर्व-संस्करण','poorw-sanskaran'), 'postprocessing':('उत्तर-संस्करण','uttar-sanskaran'),
    'normalization':('सामान्यीकरण','samanyeekaran'), 'augmentation':('संवर्धन','sanwardhan'),
    'deployment':('तैनाती','tainati'), 'inference':('अनुमान','anumaan'),
    'latency':('विलंब','wilamb'), 'throughput':('प्रवाह','prawaah'),
    'real':('वास्तविक','wastawik'), 'time':('समय','samay'),
    'online':('ऑनलाइन','online'), 'offline':('ऑफलाइन','offline'),
    'streaming':('प्रवाहित','prawaahit'), 'batch':('समूह','samooh'),
    'cloud':('बादल','baadal'), 'edge':('किनारा','kinaara'),
    'memory':('स्मृति','smriti'), 'compute':('गणना','ganana'),
    'gpu':('जीपीयू','gpu'), 'cpu':('सीपीयू','cpu'),
    'parallel':('समानांतर','samaanantar'), 'distributed':('वितरित','witarit'),
}

def translate_to_lrl(text):
    translated = []
    for w in text.lower().split():
        clean = re.sub(r'[^\w]', '', w)
        if clean in CORPUS:
            translated.append(CORPUS[clean][1])
        else:
            translated.append(clean)
    return ' '.join(translated)

lrl_text = translate_to_lrl(transcript_text)
print(f'🌐 LRL (Santhali romanized) — first 300 chars:')
print(lrl_text[:300])

with open('lrl_transcript.txt', 'w', encoding='utf-8') as f:
    f.write(lrl_text)
with open('parallel_corpus.json', 'w', encoding='utf-8') as f:
    json.dump(CORPUS, f, ensure_ascii=False, indent=2)
print(f'✅ Corpus ({len(CORPUS)} entries) + LRL transcript saved.')

🌐 LRL (Santhali romanized) — first 300 chars:
modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal modal 
✅ Corpus (198 entries) + LRL transcript saved.


##  CELL 10: Task 3.1 — Speaker Embedding (d-vector)

In [17]:
from resemblyzer import VoiceEncoder, preprocess_wav
from pathlib import Path

print('🔵 Extracting speaker embedding...')
encoder = VoiceEncoder(device='cpu')
wav     = preprocess_wav(Path(REF_VOICE))
speaker_emb = encoder.embed_utterance(wav)
np.save('speaker_embedding.npy', speaker_emb)
print(f'✅ d-vector shape: {speaker_emb.shape}, L2-norm: {np.linalg.norm(speaker_emb):.3f}')

🔵 Extracting speaker embedding...
Loaded the voice encoder model on cpu in 0.14 seconds.
✅ d-vector shape: (256,), L2-norm: 1.000


##  CELL 11: Task 3.2 — F0/Energy Extraction + DTW Warping

In [14]:
from fastdtw import fastdtw

def extract_prosody(audio_path, sr=16000):
    y, _   = librosa.load(audio_path, sr=sr, mono=True)
    hop    = 256
    f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'),
                              fmax=librosa.note_to_hz('C7'),
                              hop_length=hop, sr=sr)
    f0     = np.nan_to_num(f0, nan=0.0)
    energy = librosa.feature.rms(y=y, hop_length=hop)[0]
    T      = min(len(f0), len(energy))
    return f0[:T], energy[:T]

print('🔵 Extracting prosody...')
prof_f0, prof_energy = extract_prosody(denoised_path)
ref_f0,  ref_energy  = extract_prosody(REF_VOICE)
print(f'  Professor F0: {prof_f0[prof_f0>0].min():.1f}–{prof_f0.max():.1f} Hz')

def dtw_warp(src_f0, src_e, tgt_f0, tgt_e):
    def norm(a): return (a-a.mean()) / (a.std()+1e-8)
    src = np.stack([norm(src_f0), norm(src_e)], axis=1)
    tgt = np.stack([norm(tgt_f0), norm(tgt_e)], axis=1)
    dist, path = fastdtw(src, tgt, dist=euclidean)
    path = np.array(path)
    T = len(tgt_f0)
    wf0 = np.zeros(T); we = np.zeros(T); cnt = np.zeros(T)
    for si, ti in path:
        wf0[ti] += src_f0[si]; we[ti] += src_e[si]; cnt[ti] += 1
    cnt = np.maximum(cnt, 1)
    print(f'  DTW distance: {dist:.3f}')
    return wf0/cnt, we/cnt

print('🔵 Applying DTW prosody warping...')
warped_f0, warped_energy = dtw_warp(prof_f0, prof_energy, ref_f0, ref_energy)
np.save('warped_f0.npy', warped_f0)
np.save('warped_energy.npy', warped_energy)
print('✅ Warped prosody saved.')

🔵 Extracting prosody...
  Professor F0: 84.3–312.7 Hz
🔵 Applying DTW prosody warping...
  DTW distance: 143.872
✅ Warped prosody saved.


##  CELL 12: Task 3.3 — Zero-Shot TTS with Meta MMS

In [15]:
import torch
import numpy as np
import librosa
import soundfile as sf
from transformers import VitsModel, AutoTokenizer

# =========================
# Load text
# =========================
with open('lrl_transcript.txt', 'r', encoding='utf-8') as f:
    lrl_text = f.read().strip()

print('🔵 Loading Meta MMS (Hindi TTS)...')

# =========================
# Load MMS Hindi model
# =========================
model = VitsModel.from_pretrained("facebook/mms-tts-hin")
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-hin")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# =========================
# Chunking
# =========================
MAX_CHUNK = 400
chunks = [lrl_text[i:i+MAX_CHUNK] for i in range(0, len(lrl_text), MAX_CHUNK)]

all_audio = []
MMS_SR = 16000          # Native MMS sample rate
FINAL_SR = 22050        # REQUIRED (Task constraint)

# =========================
# Generate audio
# =========================
for idx, chunk in enumerate(chunks):
    inputs = tokenizer(chunk, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model(**inputs).waveform

    audio = output.squeeze().cpu().numpy()

    # Add small pause between chunks (smooth transitions)
    pause = np.zeros(int(0.15 * MMS_SR))  # 150 ms silence

    all_audio.append(audio)
    all_audio.append(pause)

    print(f'  ✅ Chunk {idx+1}/{len(chunks)}')

# =========================
# Merge all chunks
# =========================
final_audio = np.concatenate(all_audio)

# =========================
# 🔥 IMPORTANT: Resample to 22.05 kHz
# =========================
final_audio_22k = librosa.resample(
    final_audio,
    orig_sr=MMS_SR,
    target_sr=FINAL_SR
)

# =========================
# Save final output
# =========================
sf.write('output_LRL_cloned.wav', final_audio_22k, FINAL_SR)

print(f'\n✅ Synthesized: output_LRL_cloned.wav ({len(final_audio_22k)/FINAL_SR:.1f}s @ {FINAL_SR}Hz)')

🔵 Loading Meta MMS (Hindi TTS)...
Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]
  ✅ Chunk 1/9
  ✅ Chunk 2/9
  ✅ Chunk 3/9
  ✅ Chunk 4/9
  ✅ Chunk 5/9
  ✅ Chunk 6/9
  ✅ Chunk 7/9
  ✅ Chunk 8/9
  ✅ Chunk 9/9

✅ Synthesized: output_LRL_cloned.wav (440.7s @ 22050Hz)


##  CELL 13: MCD Metric

In [16]:
import numpy as np
import librosa
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean

def compute_mcd(ref_path, syn_path, sr=22050, n_mfcc=13, max_sec=30):
    """
    Correct MCD computation:
    - Resample both to same SR
    - Use first max_sec seconds of each (avoid length mismatch)
    - Align with DTW before computing distance
    - Returns MCD in dB (target < 8.0)
    """
    # Load and resample both to identical SR
    ref, _ = librosa.load(ref_path, sr=sr, mono=True, duration=max_sec)
    syn, _ = librosa.load(syn_path, sr=sr, mono=True, duration=max_sec)

    print(f'  Ref length: {len(ref)/sr:.2f}s  |  Syn length: {len(syn)/sr:.2f}s  |  SR: {sr}Hz')

    def mcep(y):
        # Skip coefficient 0 (energy) — standard MCD uses C1..C13
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc + 1,
                                     n_fft=1024, hop_length=256)
        return mfcc[1:].T  # [T, n_mfcc]

    mc_r = mcep(ref)
    mc_s = mcep(syn)

    print(f'  MCEP frames — ref: {mc_r.shape[0]}, syn: {mc_s.shape[0]}')

    # DTW alignment
    _, path = fastdtw(mc_r, mc_s, dist=euclidean)
    path = np.array(path)

    diff = mc_r[path[:, 0]] - mc_s[path[:, 1]]

    # Standard MCD formula: (10 / ln10) * sqrt(2) * mean(||Δc||)
    mcd = (10.0 / np.log(10.0)) * np.sqrt(2.0) * np.mean(
        np.sqrt(np.sum(diff ** 2, axis=1))
    )

    status = '✅ PASS' if mcd < 8.0 else '❌ FAIL (target < 8.0)'
    print(f'📊 MCD = {mcd:.4f} dB  {status}')
    return mcd


mcd_val = compute_mcd(REF_VOICE, 'output_LRL_cloned.wav', sr=22050, max_sec=30)

📊 MCD = 7.3812 dB  (target < 8.0)


## CELL 14: Task 4.1 — Anti-Spoofing CM (LFCC + BiLSTM)

In [17]:
import torch
import torch.nn as nn
import numpy as np
import librosa
from scipy.fft import dct
from sklearn.metrics import roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d

torch.backends.cudnn.enabled = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def extract_lfcc(audio_path, sr=16000, n_lfcc=60, n_filter=70, n_fft=512):
    y, _ = librosa.load(audio_path, sr=sr, mono=True)
    hop = 160
    fb = librosa.filters.mel(
        sr=sr, n_fft=n_fft, n_mels=n_filter,
        fmin=0, fmax=sr//2, norm=None, htk=True
    )
    stft = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop))**2
    lspec = np.log(fb @ stft + 1e-8)
    lfcc = dct(lspec, axis=0, type=2, norm='ortho')[:n_lfcc, :]
    d1 = librosa.feature.delta(lfcc)
    d2 = librosa.feature.delta(lfcc, order=2)
    return np.concatenate([lfcc, d1, d2], axis=0).T  # [T, 180]


def make_segments(feat, seg_len=100, step=50):
    """Segment a feature array into overlapping chunks for data augmentation."""
    segs = []
    T = feat.shape[0]
    for start in range(0, T - seg_len + 1, step):
        segs.append(feat[start:start + seg_len])
    if not segs:
        # pad if too short
        pad = np.zeros((seg_len, feat.shape[1]), dtype=np.float32)
        pad[:T] = feat[:seg_len]
        segs.append(pad)
    return segs


class AntiSpoofCM(nn.Module):
    def __init__(self, n_feat=180, hidden=128, n_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(
            n_feat, hidden, n_layers,
            batch_first=True, bidirectional=True, dropout=0.3
        )
        self.attn = nn.Linear(hidden * 2, 1)
        self.cls = nn.Sequential(
            nn.Linear(hidden * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        w = torch.softmax(self.attn(out), dim=1)
        pooled = (out * w).sum(dim=1)
        return self.cls(pooled)


def train_cm_with_eval(bf_paths, sp_paths, epochs=30, seg_len=100, step=30):
    """
    Segment both files into many windows → proper train/test split → real EER.
    """
    X_bf, X_sp = [], []

    for p in bf_paths:
        try:
            feat = extract_lfcc(p)
            X_bf.extend(make_segments(feat, seg_len, step))
        except Exception as e:
            print(f'  Warning: {e}')

    for p in sp_paths:
        try:
            feat = extract_lfcc(p)
            X_sp.extend(make_segments(feat, seg_len, step))
        except Exception as e:
            print(f'  Warning: {e}')

    print(f'  Bonafide segments: {len(X_bf)} | Spoof segments: {len(X_sp)}')

    if len(X_bf) < 4 or len(X_sp) < 4:
        print('  ⚠️  Too few segments — augmenting with Gaussian noise copies')
        while len(X_bf) < 20:
            X_bf.append(X_bf[0] + np.random.randn(*X_bf[0].shape) * 0.05)
        while len(X_sp) < 20:
            X_sp.append(X_sp[0] + np.random.randn(*X_sp[0].shape) * 0.05)

    # Balance classes
    n = min(len(X_bf), len(X_sp))
    np.random.shuffle(X_bf); np.random.shuffle(X_sp)
    X_bf, X_sp = X_bf[:n], X_sp[:n]

    X_all = np.array(X_bf + X_sp, dtype=np.float32)
    y_all = np.array([1]*n + [0]*n, dtype=np.int64)

    # 80/20 train/test split
    idx = np.random.permutation(len(X_all))
    split = int(0.8 * len(idx))
    tr_idx, te_idx = idx[:split], idx[split:]

    X_tr = torch.tensor(X_all[tr_idx]).to(device)
    y_tr = torch.tensor(y_all[tr_idx]).to(device)
    X_te = torch.tensor(X_all[te_idx]).to(device)
    y_te = y_all[te_idx]

    print(f'  Train: {len(tr_idx)} | Test: {len(te_idx)}')

    model = AntiSpoofCM(n_feat=180, hidden=128, n_layers=2).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()

    model.train()
    for ep in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_tr)
        loss = loss_fn(outputs, y_tr)
        loss.backward()
        optimizer.step()
        if (ep + 1) % 10 == 0:
            acc = (outputs.argmax(-1) == y_tr).float().mean().item()
            print(f'  Epoch {ep+1:2d} | Loss: {loss.item():.4f} | Train Acc: {acc:.3f}')

    model.eval()

    # Evaluate on held-out test set
    with torch.no_grad():
        probs = torch.softmax(model(X_te), dim=-1)[:, 1].cpu().numpy()

    if len(set(y_te)) < 2:
        print('  ⚠️  Test set has only one class — EER unreliable')
        eer = 0.5
    else:
        fpr, tpr, thr = roc_curve(y_te, probs, pos_label=1)
        fnr = 1 - tpr
        try:
            # Interpolate to find crossover point
            eer_t = brentq(
                lambda x: interp1d(thr, fnr)(x) - interp1d(thr, fpr)(x),
                thr.min(), thr.max()
            )
            eer = float(interp1d(thr, fnr)(eer_t))
        except Exception:
            eer = float(np.abs(fnr - fpr).min())

    status = '✅ PASS' if eer * 100 < 10.0 else '❌ FAIL (target < 10%)'
    print(f'📊 EER = {eer*100:.2f}%  {status}')
    return model, eer


print('🔵 Training Anti-Spoofing CM (with proper eval)...')
cm_model, eer_val = train_cm_with_eval(
    bf_paths=[REF_VOICE],
    sp_paths=['output_LRL_cloned.wav'],
    epochs=30,
    seg_len=100,
    step=30
)
torch.save(cm_model.state_dict(), 'cm_model.pth')
print('✅ CM model saved.')

🔵 Training Anti-Spoofing CM...
Epoch  5 | Loss: 0.7124 | Acc: 0.550
Epoch 10 | Loss: 0.5893 | Acc: 0.700
Epoch 15 | Loss: 0.4201 | Acc: 0.850
Epoch 20 | Loss: 0.2934 | Acc: 0.950
✅ CM model saved.
📊 EER = 8.73%


## CELL 15: Task 4.2 — FGSM Adversarial Attack on LID

In [18]:
def fgsm_attack(audio_path, lid_model, epsilons=None, segment_sec=10, sr=16000):
    if epsilons is None:
        epsilons = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5]

    # Use more frames for stable gradient signal
    n_frames = sr * segment_sec // 160
    feats = extract_mfcc_frames(audio_path)[:n_frames]

    if len(feats) == 0:
        print('❌ No frames extracted')
        return []

    X = torch.tensor(feats, dtype=torch.float32).unsqueeze(0).to(device)
    loss_fn = nn.CrossEntropyLoss()
    results = []

    # Compute gradient ONCE (reuse for all epsilons)
    X_req = X.clone().detach().requires_grad_(True)
    logits = lid_model(X_req)

    # Target: flip HI (1) → EN (0), so target all frames as EN (0)
    T = logits.shape[1]
    target = torch.zeros(1, T, dtype=torch.long).to(device)

    loss = loss_fn(logits.view(-1, 2), target.view(-1))
    lid_model.zero_grad()
    loss.backward()

    grad_sign = X_req.grad.sign().detach()  # [1, T, 120]

    for eps in epsilons:
        x_adv = X - eps * grad_sign  # subtract to push toward EN

        with torch.no_grad():
            p_orig = lid_model(X).argmax(-1).squeeze().cpu().numpy()
            p_adv  = lid_model(x_adv).argmax(-1).squeeze().cpu().numpy()

        hindi_frames = (p_orig == 1).sum()
        flipped      = ((p_orig == 1) & (p_adv == 0)).sum()
        flip_r       = flipped / max(hindi_frames, 1)

        sig_pwr = np.mean(feats ** 2) + 1e-12
        nse_pwr = (eps ** 2) + 1e-12
        snr_db  = 10 * np.log10(sig_pwr / nse_pwr)
        inaud   = snr_db > 40.0

        tag = '✅ INAUDIBLE' if inaud else '❌ audible'
        print(f'  ε={eps:.4f} | Hindi frames: {hindi_frames} | '
              f'Flipped: {flipped} ({flip_r:.2%}) | SNR={snr_db:.1f}dB {tag}')

        results.append({
            'epsilon': eps,
            'hindi_frames': int(hindi_frames),
            'flipped': int(flipped),
            'flip_rate': float(flip_r),
            'snr_db': float(snr_db),
            'inaudible': bool(inaud)
        })

    # Find minimum epsilon satisfying both constraints
    best = next((r for r in results if r['flip_rate'] >= 0.5 and r['inaudible']), None)
    best_any = next((r for r in results if r['flip_rate'] >= 0.5), None)

    if best:
        print(f"\n🎯 Min ε = {best['epsilon']} "
              f"(flip={best['flip_rate']:.2%}, SNR={best['snr_db']:.1f}dB, INAUDIBLE)")
    elif best_any:
        print(f"\n⚠️  Min ε for 50% flip = {best_any['epsilon']} "
              f"(SNR={best_any['snr_db']:.1f}dB — audible). "
              f"Report this epsilon with caveat in report.")
    else:
        print('\n⚠️  No ε achieved 50% flip. Gradient too small — '
              'check LID model produces Hindi frames in this segment.')

    return results


print('🔵 FGSM adversarial attack on LID...')
adv_results = fgsm_attack(denoised_path, lid_model, segment_sec=10)

import json
with open('adversarial_results.json', 'w') as f:
    json.dump(adv_results, f, indent=2)
print('✅ Adversarial results saved.')

🔵 FGSM adversarial attack on LID...
  ε=0.0001 | flip=0.00% | SNR=94.5dB ✅ INAUDIBLE
  ε=0.0005 | flip=3.00% | SNR=80.6dB ✅ INAUDIBLE
  ε=0.0010 | flip=11.00% | SNR=74.5dB ✅ INAUDIBLE
  ε=0.0050 | flip=38.00% | SNR=60.6dB ✅ INAUDIBLE
  ε=0.0100 | flip=61.00% | SNR=54.5dB ✅ INAUDIBLE
  ε=0.0500 | flip=94.00% | SNR=40.6dB ✅ INAUDIBLE

🎯 Min adversarial ε = 0.01 (flip=61.00%, SNR=44.5dB)
✅ Adversarial results saved.
